In [39]:
import json
import requests
import openai
from dotenv import load_dotenv

load_dotenv()

client = openai.OpenAI()
messages = []

In [40]:
default_url = "https://nomad-movies.nomadcoders.workers.dev/"

In [41]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "A function to get popular movie list",
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "A function to get movie information with id",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "key value of movie, that represents specific movie"
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "A function to get movie information which similar with movie searching by id",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "key value of movie, that represents specific movie"
                    }
                },
                "required": ["id"]
            }
        }
    }
]

In [42]:
from typing import TypedDict


class Movie(TypedDict):
    adult: bool
    backdrop_path: str
    genre_ids: list[int]
    id: int
    original_language: str
    original_title: str
    overview: str
    popularity: float
    poster_path: str
    release_date: str
    title: str
    video: bool
    vote_average: float
    vote_count: int


Movies = list[Movie]

In [43]:
def get_popular_movies():
    url = default_url + "movies"
    response: Movies = requests.get(url).json()

    result = []

    for movie in response:
        result.append({
            "id": movie["id"],
            "title": movie["title"],
        })

    return str(result)


def get_movie_details(id: str):
    url = default_url + "movies/" + id
    return str(requests.get(url).json())


def get_similar_movies(id: str):
    url = default_url + "movies/" + id + "/similar"
    response: Movies = requests.get(url).json()

    result = []

    for movie in response:
        result.append({
            "id": movie["id"],
            "title": movie["title"],
        })

    return str(result)

In [44]:
FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_similar_movies": get_similar_movies
}

In [45]:
def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS
    )

    process_ai_response(response.choices[0].message)

In [46]:
from openai.types.chat import ChatCompletionMessage


def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments
                        }
                    } for tool_call in message.tool_calls
                ]
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Agent: [{function_name}({arguments}) called]")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": result
            })

        call_ai()
    else:
        messages.append(
            {
                "role": "assistant",
                "content": message.content
            }
        )
        print(f"Agent: {message.content}")

In [47]:
while True:
    message = input("Send a message to the LLM...")

    if message == "quit" or message == "q":
        break
    else:
        messages.append(
            {
                "role": "user",
                "content": message
            }
        )
        print(f"User: {message}")
        call_ai()

User: 지금 인기 있는 영화 알려줘.
Agent: [get_popular_movies({}) called]
Agent: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Shelter**
2. **The Bluff**
3. **Mercy**
4. **Hellfire**
5. **Your Heart Will Be Broken**
6. **The Orphans**
7. **Scream 7**
8. **Return to Silent Hill**
9. **Turbulence**
10. **A Woman Scorned**
11. **The Shadow's Edge**
12. **Dog 51**
13. **Zootopia 2**
14. **28 Years Later: The Bone Temple**
15. **Hierarchy**
16. **Greenland 2: Migration**
17. **The Housemaid**
18. **Avatar: Fire and Ash**
19. **Pose**
20. **In the Morning of La Petite Mort**

더 궁금한 영화가 있으면 말씀해 주세요!
User: 듄에 대해 더 알려줘.


BadRequestError: Error code: 400 - {'error': {'message': "Invalid value for 'content': expected a string, got null.", 'type': 'invalid_request_error', 'param': 'messages.[5].content', 'code': None}}